<a href="https://colab.research.google.com/github/yaelezra/ReportAgent/blob/main/ReportAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install smolagents huggingface_hub

In [ ]:
!pip install sentence-transformers

In [ ]:
import scipy.io as sio
import pandas as pd
from google.colab import userdata
from smolagents import tool, CodeAgent, InferenceClientModel
import numpy as np
from google.colab import output

In [ ]:
@tool
def load_mat_file(file_path: str) -> np.array:
    """
    Loads a self-documenting MATLAB sequence file.
    Returns np array of the sequence data.

    Args:
      file_path: The string path to your .mat file.
    """
    # 1. Load the file cleanly using our magic arguments
    mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)

    # 2. Extract the main struct and the parameter values
    seq = mat_contents['sequence_data']

    return seq

In [ ]:
@tool
def load_sequence(file_path: str, param_name: str) -> np.array:
    """
    Loads a self-documenting MATLAB sequence file.
    Returns the sequence data for a specific parameter (param_name).

    Args:
      file_path: The string path to your .mat file.
      param_name: The name of the parameter you want to extract.
    """
    # 1. Load the file cleanly using our magic arguments
    mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)

    # 2. Extract the main struct and the parameter values
    seq = mat_contents['sequence_data']
    seq_param = getattr(seq, param_name)

    return seq_param

In [ ]:
@tool
def load_doc(file_path: str) -> dict:
      """
      Loads a self-documenting MATLAB sequence file.
      Returns the documentation of all parameters.

      Args:
        file_path: The string path to your .mat file.
      """
      # 1. Load the file cleanly using our magic arguments
      mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)

      # 2. Extract the main struct
      seq = mat_contents['sequence_data']

      # 3. Parse the documentation into a readable string
      doc_header = f"--- Documentation for {file_path} ---\n"
      doc_entries = {}

      for field in seq.documentation._fieldnames:
          description = getattr(seq.documentation, field)
          doc_entries[field] = description

      return doc_entries

In [ ]:
from sentence_transformers import SentenceTransformer, util
embedder = SentenceTransformer('all-MiniLM-L6-v2')

@tool
def find_feature_by_description(file_path: str, query: str, top_k: int = 3) -> dict:
    """
    Finds the most semantically similar features to the query using embedding similarity.
    Use this when the user asks for a feature by a loose or natural language name.
    Returns the top matching field names, their descriptions, and similarity scores.

    Args:
      file_path: The string path to your .mat file.
      query: A natural language description of the feature you're looking for.
      top_k: Number of top matches to return (default 3).
    """
    mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)
    seq = mat_contents['sequence_data']

    # Build corpus: "field_name: description"
    fields, corpus = [], []
    for field in seq._fieldnames:
        if field == 'documentation':
            continue
        description = ''
        if field in seq.documentation._fieldnames:
            description = str(getattr(seq.documentation, field))
        fields.append(field)
        corpus.append(f"{field}: {description}")

    query_emb  = embedder.encode(query, convert_to_tensor=True)
    corpus_embs = embedder.encode(corpus, convert_to_tensor=True)

    scores = util.cos_sim(query_emb, corpus_embs)[0]
    top = scores.topk(min(top_k, len(fields)))

    results = {}
    for score, idx in zip(top.values, top.indices):
        field = fields[idx]
        results[field] = {
            "description": corpus[idx],
            "similarity":  round(float(score), 3)
        }

    return results

In [ ]:
@tool
def find_feature_in_files(file_paths: list, param_name: str) -> list:
  """
  Find the same feature value in different files.

  Args:
    file_paths: TThe list of the mat files.
    param_name: The name of the parameter you want to extract.
  """
  file_features = []
  for path in file_paths:
    seq = load_mat_file(path)
    file_features.append(getattr(seq, param_name).to_list())
  return file_features



In [ ]:
@tool
def list_all_features(file_path: str) -> dict:
    """
    Returns all available numeric field names and their descriptions.
    Always call this first to know what's in the file.
    Args:
      file_path: The string path to your .mat file.
    """
    mat = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)
    seq = mat['sequence_data']
    doc = {f: str(getattr(seq.documentation, f)) for f in seq.documentation._fieldnames}
    return {
        f: doc.get(f, '')
        for f in seq._fieldnames
        if f != 'documentation'
        and isinstance(getattr(seq, f), np.ndarray)
        and np.issubdtype(getattr(seq, f).dtype, np.number)
    }

In [ ]:
# Assuming your tools and model are already initialized from the previous step...

# Load your secure token
hf_token = userdata.get('HF_TOKEN')

# Initialize the model
model = InferenceClientModel(model_id="meta-llama/Llama-3.3-70B-Instruct", token=hf_token)

agent = CodeAgent(
    tools=[load_mat_file, load_sequence, load_doc, find_feature_by_description, find_feature_in_files],
    model=model,
    additional_authorized_imports=["numpy", "pandas", "seaborn", "matplotlib.pyplot", "math"]
)

# Keep a history list outside the function
conversation_history = []

def run_custom_agent(file_paths: list, instructions: str, user_prompt: str):
    """
    Feeds strict instructions, conversation history, and a user prompt to the agent.
    """
    # Build history string from previous exchanges
    history_str = "\n".join(
        f"Q: {h['q']}\nA: {h['a']}"
        for h in conversation_history[-6:]  # last 6 exchanges
    ) if conversation_history else "None."

    full_prompt = f"""
    You are given a list of files - each file contains a struct in which each field is a vector of values in time.
    {file_paths} is the list of file paths.

    CONVERSATION HISTORY (use this to answer follow-up questions):
    {history_str}

    SYSTEM INSTRUCTIONS TO FOLLOW STRICTLY:
    {instructions}

    --------------------------------------------------
    USER REQUEST:
    {user_prompt}
    """

    print("🤖 Agent is thinking...\n")
    response = agent.run(full_prompt)

    # Save this exchange to history
    conversation_history.append({"q": user_prompt, "a": str(response)})

    print("\n🎯 FINAL ANSWER:")
    print(response)
    return response

In [ ]:
# ==========================================
# HOW TO USE IT
# ==========================================

# 1. Define your strict rules (You can change these whenever you want!)
my_rules = """
- You are a senior Data Scientist.
- You know how to analyze big data, give insights, conclusions, and make graphs and reports.
- Always call list_all_features first to understand what fields are available.
- If you are not sure of a field name, use find_feature_by_description before trying to load it.
- In your final answer, tell which tools you used to answer the question.
- Ellaborate your final answer as much as you can.
- If you encounter an error that you can't resolve, stop and print the error as your response.
- If the user asks you a report on something you should make it in html format and save it.
- If you make plots, graphs or reports, show them.
- Always debug your code before executing.
"""

# 2. Define the specific question you want to ask right now
my_question = "Can you do the same statics for the second file and than make a report in html that comapers statistics of happiness level in both files?"

# 3. Run the agent with both!
file_paths = ['/content/daily_life_analytics1.mat', '/content/daily_life_analytics2.mat']
response = run_custom_agent(file_paths, instructions=my_rules, user_prompt=my_question)